# 01 · Extract Overture (DuckDB → S3 → GeoParquet)

Runs the same code as `make extract` (`python -m urban_access.cli extract`), then verifies every
output: row count, CRS, bounds, null/invalid geometries, plus a plot. The extent is the city bbox ∪ the
envelope of the scored block groups (centroid inside the bbox), buffered by `edge_buffer_m`.

In [ ]:
import os

import urban_access  # noqa: F401  (first: sets PROJ_DATA before geopandas/pyogrio load)
from urban_access.config import load_city

CITY = os.environ.get("CITY", "carson_city")
cfg = load_city(CITY)
cfg.ensure_dirs()
print(
    cfg.name, "| Overture", cfg.overture_release, "| ACS", cfg.acs_year, "| EPSG", cfg.analysis_epsg
)
print("raw:", cfg.raw_dir)

In [ ]:
import json

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from shapely.geometry import box

from urban_access import census, extract

census.run(cfg)  # block groups define the extent
bg = gpd.read_parquet(cfg.raw_dir / "block_groups.parquet")
scored_ext = census.scored_extent(cfg)
extent = scored_ext.buffered(cfg.edge_buffer_m, cfg.analysis_epsg)
print("scored block groups:", int(bg["scored"].sum()), "of", len(bg))
print("bbox      ", cfg.bbox.as_tuple())
print("scored ext", tuple(round(v, 4) for v in scored_ext.as_tuple()))
print("fetch ext ", tuple(round(v, 4) for v in extent.as_tuple()))

In [ ]:
paths = extract.extract_all(cfg, extent)  # cached: pass force=True to re-download
paths

## Verification

In [ ]:
ext_poly = box(*extent.as_tuple())
rows = []
layers = {}
for name, path in paths.items():
    g = gpd.read_parquet(path)
    layers[name] = g
    meta = json.loads(path.with_suffix(".meta.json").read_text())
    xmin, ymin, xmax, ymax = g.total_bounds
    rows.append(
        {
            "file": path.name,
            "rows": len(g),
            "crs": g.crs.to_string() if g.crs else None,
            "wgs84_lonlat": g.crs is not None
            and (g.crs.equals("OGC:CRS84") or g.crs.to_epsg() == 4326),
            "null_geom": int(g.geometry.isna().sum()),
            "invalid_geom": int((~g.geometry.is_valid).sum()),
            "intersects_extent": bool(g.intersects(ext_poly).all()),
            "bounds": (round(xmin, 3), round(ymin, 3), round(xmax, 3), round(ymax, 3)),
            "release": meta["release"],
            "MB": round(path.stat().st_size / 1e6, 2),
        }
    )
check = pd.DataFrame(rows).set_index("file")
check

In [ ]:
assert (check["rows"] > 0).all(), "empty extract"
assert (check["null_geom"] == 0).all(), "null geometries"
assert check["intersects_extent"].all(), "features outside extent"
assert (check["release"] == cfg.overture_release).all()
assert check["wgs84_lonlat"].all()  # OGC:CRS84 = EPSG:4326 with lon/lat order
# invalid polygons (buildings / land use) get ST_MakeValid after loading into PostGIS
check["invalid_geom"]

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
style = {
    "buildings": dict(color="gray", linewidth=0),
    "segments": dict(color="black", linewidth=0.3),
    "connectors": dict(color="red", markersize=0.2),
    "places": dict(color="blue", markersize=1),
    "parks": dict(color="green"),
    "neighborhoods": dict(facecolor="none", edgecolor="purple"),
}
for ax, (name, g) in zip(axes.flat, layers.items(), strict=False):
    bg.boundary.plot(ax=ax, color="orange", linewidth=0.4)
    g.plot(ax=ax, **style.get(name, {}))
    gpd.GeoSeries([box(*cfg.bbox.as_tuple())], crs=4326).boundary.plot(ax=ax, color="k", ls="--")
    ax.set_title(f"{name} ({len(g):,})")
    ax.set_xlim(extent.lon_min, extent.lon_max)
    ax.set_ylim(extent.lat_min, extent.lat_max)
plt.tight_layout()

## Attribute distributions for Phase 2 decisions

Walkable-segment filter (open item 2) and the residential-building heuristic (open item 3).

In [ ]:
seg = layers["segments"]
print(
    seg.groupby(["class", "subclass"], dropna=False).size().sort_values(ascending=False).to_string()
)
print("\nwith access_restrictions:", seg["access_restrictions_json"].notna().sum())
print("with road_flags:", seg["road_flags_json"].notna().sum())
seg.loc[seg["access_restrictions_json"].notna(), ["class", "access_restrictions_json"]].head(10)

In [ ]:
bld = layers["buildings"].to_crs(cfg.analysis_epsg)
bld["area_m2"] = bld.area
summary = (
    bld.groupby(["subtype", "class"], dropna=False)
    .agg(
        n=("id", "size"),
        median_area_m2=("area_m2", "median"),
        has_floors=("num_floors", lambda s: s.notna().mean()),
        has_height=("height", lambda s: s.notna().mean()),
    )
    .sort_values("n", ascending=False)
)
print(summary.round(2).head(25).to_string())
uncl = bld[bld["subtype"].isna() & bld["class"].isna()]["area_m2"]
print("\nunclassified footprint area quantiles (m²):")
print(uncl.quantile([0.05, 0.25, 0.5, 0.75, 0.95]).round(0))

In [ ]:
print(layers["parks"].groupby(["subtype", "class"]).size())
print(layers["neighborhoods"][["name", "subtype"]])